In [1]:
message = "Han, Mitakuyapi!"
print(message)

Han, Mitakuyapi!


In [2]:
# ============================================================
# NOTEBOOK 13 — CELL 1
# ENVIRONMENTAL DATA CUBE FOR OGLALA LAKOTA
# Inventory existing datasets for dissertation map atlas
#
# MAP 7: Sampling lattice
# MAP 5: NDVI through time
# MAP 6: Bare soil / NDBSI
#
# READ-ONLY: Does not change existing files
# ============================================================

from pathlib import Path
import xarray as xr
import rasterio
import geopandas as gpd

# ------------------------------------------------------------
# 1. PROJECT DIRECTORIES
# ------------------------------------------------------------

project = (
    Path.home()
    / "Documents/ElishaWakinyanZi-WORK"
    / "datacube_2026/ate_makoce_agcube"
)

data_dir = project / "data"
output_dir = project / "outputs"
results_dir = project / "results"

print("=" * 70)
print("ENVIRONMENTAL DATA CUBE FOR OGLALA LAKOTA")
print("DISSERTATION MAP ATLAS — DATA INVENTORY")
print("=" * 70)

print("\nProject:", project)
print("Project exists:", project.exists())

if not project.exists():
    raise FileNotFoundError(
        f"Project directory not found: {project}"
    )

# ------------------------------------------------------------
# 2. FIND NETCDF DATA CUBES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. NETCDF DATA CUBES")
print("=" * 70)

nc_files = sorted(project.rglob("*.nc"))

print(f"\nFound {len(nc_files)} NetCDF files.")

for i, path in enumerate(nc_files, 1):

    print(f"\n[{i}] {path.relative_to(project)}")

    try:
        with xr.open_dataset(path) as ds:

            print("  Dimensions:", dict(ds.sizes))
            print("  Variables:", list(ds.data_vars))
            print("  Coordinates:", list(ds.coords))

            if "title" in ds.attrs:
                print("  Title:", ds.attrs["title"])

    except Exception as e:
        print("  Could not inspect:", str(e))

# ------------------------------------------------------------
# 3. FIND EXISTING GEOTIFF RASTERS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. GEOTIFF RASTERS")
print("=" * 70)

tif_files = sorted(
    list(project.rglob("*.tif"))
    + list(project.rglob("*.tiff"))
)

print(f"\nFound {len(tif_files)} GeoTIFF files.")

keywords = [
    "ndvi",
    "ndmi",
    "ndbsi",
    "evi",
    "trajectory",
    "vegetation",
    "soil",
    "landsat"
]

relevant_tifs = [
    p for p in tif_files
    if any(
        word in p.name.lower()
        for word in keywords
    )
]

print(
    f"Potentially relevant rasters: "
    f"{len(relevant_tifs)}"
)

for i, path in enumerate(relevant_tifs, 1):

    print(f"\n[{i}] {path.relative_to(project)}")

    try:
        with rasterio.open(path) as src:

            print("  Shape:", (src.height, src.width))
            print("  Bands:", src.count)
            print("  CRS:", src.crs)
            print("  Resolution:", src.res)

    except Exception as e:
        print("  Could not inspect:", str(e))

# ------------------------------------------------------------
# 4. FIND PARCEL AND SAMPLING SHAPEFILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. SPATIAL VECTOR DATA")
print("=" * 70)

shp_files = sorted(project.rglob("*.shp"))

print(f"\nFound {len(shp_files)} shapefiles.")

for i, path in enumerate(shp_files, 1):

    print(f"\n[{i}] {path.relative_to(project)}")

    try:
        gdf = gpd.read_file(path)

        print("  Features:", len(gdf))
        print("  Geometry:", list(gdf.geom_type.unique()))
        print("  CRS:", gdf.crs)

    except Exception as e:
        print("  Could not inspect:", str(e))

# ------------------------------------------------------------
# 5. IDENTIFY LIKELY MAP INPUTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. MAP INPUT CANDIDATES")
print("=" * 70)

map_keywords = {
    "MAP 7 — SAMPLING LATTICE": [
        "lattice", "point", "wanzi", "parcel"
    ],
    "MAP 5 — NDVI THROUGH TIME": [
        "ndvi", "2019", "2022", "2023"
    ],
    "MAP 6 — BARE SOIL / NDBSI": [
        "ndbsi", "bare", "soil"
    ]
}

all_files = nc_files + relevant_tifs + shp_files

for map_name, terms in map_keywords.items():

    print(f"\n{map_name}")

    matches = [
        p for p in all_files
        if any(
            term in p.name.lower()
            for term in terms
        )
    ]

    if matches:

        for p in matches[:20]:
            print("  ", p.relative_to(project))

        if len(matches) > 20:
            print(
                f"  ... and {len(matches) - 20} more"
            )

    else:
        print("  No matching filenames found.")

# ------------------------------------------------------------
# 6. INVENTORY SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INVENTORY COMPLETE")
print("=" * 70)

print("NetCDF files:", len(nc_files))
print("GeoTIFF files:", len(tif_files))
print("Relevant GeoTIFFs:", len(relevant_tifs))
print("Shapefiles:", len(shp_files))

print("\nNo original datasets were modified.")

ENVIRONMENTAL DATA CUBE FOR OGLALA LAKOTA
DISSERTATION MAP ATLAS — DATA INVENTORY

Project: /Users/elishawakinyanzi/Documents/ElishaWakinyanZi-WORK/datacube_2026/ate_makoce_agcube
Project exists: True

1. NETCDF DATA CUBES

Found 14 NetCDF files.

[1] Notebooks/environmental_data_cube_oglala_lakota_2000_2023.nc
  Dimensions: {'year': 24}
  Variables: ['ndvi', 'tmax_k', 'precip_mm', 'evi', 'evi_previous', 'evi_screened', 'evi_valid_composites', 'evi_low_coverage_composites']
  Coordinates: ['year']
  Title: Environmental Data Cube for Oglala Lakota

[2] Notebooks/environmental_data_cube_oglala_lakota_dissertation.nc
  Dimensions: {'year': 3}
  Variables: ['landsat_parcel_ndvi', 'landsat_parcel_ndmi', 'reservation_evi', 'growing_precip_mm', 'growing_tmax_c', 'parcel_pixel_count']
  Coordinates: ['year']
  Title: Environmental Data Cube for Oglala Lakota

[3] data/cache/environmental_data_cube_oglala_lakota_2000_2023_validated_evi_v2.nc
  Dimensions: {'year': 24}
  Variables: ['ndvi', 'tm

In [ ]:

# ============================================================
# NOTEBOOK 13 — CELL 2
# ENVIRONMENTAL DATA CUBE FOR OGLALA LAKOTA
#
# Verify spatial data sources for:
# MAP 7 — Sampling lattice
# MAP 5 — NDVI through time
# MAP 6 — Bare soil / NDBSI
#
# READ-ONLY: No original files modified
# ============================================================

from pathlib import Path
import numpy as np
import xarray as xr
import rasterio
import geopandas as gpd

from rasterio.warp import transform_bounds

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

project = (
    Path.home()
    / "Documents/ElishaWakinyanZi-WORK"
    / "datacube_2026/ate_makoce_agcube"
)

parcel_path = project / "data/ate_parcel_wanzi.shp"

ndbsi_path = (
    project
    / "data/cache/ndbsi_comparison_2019_2023"
    / "pine_ridge_ndbsi_2019.tif"
)

spatial_cube_path = (
    project
    / "data/cache/spatial_grid_0.5deg_2000_2023.nc"
)

print("=" * 65)
print("NOTEBOOK 13 — SPATIAL DATA VERIFICATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. PARCEL GEOMETRY
# ------------------------------------------------------------

parcel = gpd.read_file(parcel_path)

print("\n1. ATE_MACOKE PARCEL")
print("-" * 45)

print("Features:", len(parcel))
print("Original CRS:", parcel.crs)

parcel_utm = parcel.to_crs("EPSG:32613")

print("Projected CRS:", parcel_utm.crs)
print("Bounds (UTM):", parcel_utm.total_bounds)

parcel_area_ha = (
    parcel_utm.geometry.area.sum() / 10000
)

print(f"Parcel area: {parcel_area_ha:.2f} hectares")

# ------------------------------------------------------------
# 3. CHECK NDBSI RASTER COVERAGE
# ------------------------------------------------------------

print("\n2. NDBSI RASTER COVERAGE")
print("-" * 45)

with rasterio.open(ndbsi_path) as src:

    print("Raster:", ndbsi_path.name)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Dimensions:", src.width, "x", src.height)
    print("NoData:", src.nodata)

    raster_bounds = src.bounds
    print("Raster bounds:", raster_bounds)

    # Reproject parcel into raster CRS.
    parcel_raster_crs = parcel.to_crs(src.crs)

    # Raster footprint
    from shapely.geometry import box

    raster_footprint = box(*raster_bounds)

parcel_geometry = parcel_raster_crs.geometry.iloc[0]
 
    intersection = (
        parcel_geometry.intersection(raster_footprint)
    )

    parcel_area = parcel_geometry.area

    coverage_pct = (
        100 * intersection.area / parcel_area
        if parcel_area > 0 else 0
    )

    print(
        f"Parcel footprint coverage: "
        f"{coverage_pct:.2f}%"
    )

    # Check actual valid raster data inside the parcel.
    from rasterio.mask import mask
    from rasterio.errors import WindowError

    if intersection.is_empty:

        print("No overlap with Ate_Macoke.")

    else:

        clipped, clipped_transform = mask(
            src,
            parcel_raster_crs.geometry,
            crop=True,
            filled=False
        )

        band = clipped[0]
        valid = band.compressed()

        print(
            "Valid raster pixels inside parcel:",
            len(valid)
        )

        if len(valid) > 0:

            print(
                "NDBSI minimum:",
                float(np.min(valid))
            )

            print(
                "NDBSI maximum:",
                float(np.max(valid))
            )

            print(
                "NDBSI mean:",
                float(np.mean(valid))
            )

        else:
            print(
                "Raster footprint overlaps parcel, "
                "but no valid NDBSI pixels were found."
            )

# ------------------------------------------------------------
# 4. INSPECT SPATIAL NDVI CUBE
# ------------------------------------------------------------

print("\n3. SPATIAL NDVI CUBE")
print("-" * 45)

with xr.open_dataset(spatial_cube_path) as ds:

    print("Title:", ds.attrs.get("title"))
    print("Dimensions:", dict(ds.sizes))
    print("Variables:", list(ds.data_vars))

    print("\nLatitude coordinates:")
    print(ds["lat"].values)

    print("\nLongitude coordinates:")
    print(ds["lon"].values)

    print("\nYears:")
    print(ds["year"].values)

    print("\nNDVI attributes:")
    print(ds["ndvi"].attrs)

    print("\nDataset attributes:")
    print(ds.attrs)

# ------------------------------------------------------------
# 5. SEARCH FOR SAMPLING LATTICE FILES
# ------------------------------------------------------------

print("\n4. SEARCH FOR SAMPLING LATTICE")
print("-" * 45)

# Search by filename across the wider datacube workspace.
workspace = project.parent

extensions = {
    ".nc", ".csv", ".geojson",
    ".gpkg", ".parquet", ".json", ".shp"
}

search_terms = [
    "lattice",
    "sampling",
    "15pts",
    "15_points",
    "wanzi",
    "point_grid"
]

matches = []

for path in workspace.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    if any(
        term in path.name.lower()
        for term in search_terms
    ):
        matches.append(path)

print(f"Candidate sampling files: {len(matches)}")

for path in sorted(matches)[:60]:

    print(path.relative_to(workspace))

if len(matches) > 60:
    print(f"... plus {len(matches) - 60} more")

# ------------------------------------------------------------
# 6. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SPATIAL VERIFICATION COMPLETE")
print("=" * 65)

print(
    "\nNext: Determine which existing datasets "
    "can support parcel-scale maps."
)

print("Original datasets remain unchanged.")


IndentationError: unindent does not match any outer indentation level (<string>, line 97)